# DDPG + SLM Sentiment Portfolio Workflow

This notebook runs the experimental SLM extension.

The offline model is still a DDPG model. The SLM signal is added during online evaluation.


## 0. Import shared workflow API

- Keep the notebook readable.
- Keep reusable code in `src/finance_rl_slm/`.
- Use the same output folders as the pure DDPG notebook.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in (PROJECT_ROOT / "src", PROJECT_ROOT, PROJECT_ROOT / "src" / "FinRL"):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from finance_rl_slm.config import DEFAULT_CONFIG
from finance_rl_slm.workflow import (
    build_daily_sentiment,
    build_sentiment_inputs,
    load_online_price_data,
    load_price_data,
    plot_normalized_prices,
    print_runtime_context,
    result_picture_path,
    result_profile_path,
    split_price_data,
)
from finance_rl_slm.evaluation import run_online_evaluation
from finance_rl_slm.training import train_offline_model


## 1. Configure the experiment

- Online test window: `2026-01-01` to `2026-06-21`.
- Pipeline name: `ddpg_slm`.
- News sample: `news_max_items = 3` per ticker for a practical local Granite run.
- SLM sentiment is treated as an experimental extra feature.


In [ ]:
config = replace(
    DEFAULT_CONFIG,
    online_start="2026-01-01",
    online_end="2026-06-21",
    news_max_items=3,
)
print_runtime_context(config)


## 2. Build weekly SLM sentiment

- Pull Yahoo Finance RSS news for each ticker.
- Use IBM Granite to score news sentiment.
- Aggregate the signal by week and then align it to trading days.


In [ ]:
df_all_news, weekly, weekly_mkt = build_sentiment_inputs(config)


## 3. Download price data and train the offline DDPG model

- Training is still price-based.
- SLM is not used during offline training.
- Running this cell can take a long time.


In [ ]:
price_df = load_price_data(config)
plot_normalized_prices(price_df, config)

splits = split_price_data(price_df, config)
model = train_offline_model(splits.train, splits.valid, config)


## 4. Run online evaluation with SLM sentiment

- Convert weekly sentiment into a daily `sentiment_series`.
- Pass it into the online environment.
- Save plots and profile CSV files under `addenda/`.


In [ ]:
price_df_online = load_online_price_data(config.online_end, config)
sentiment_series = build_daily_sentiment(weekly_mkt, price_df_online)

ddpg_slm_logs = run_online_evaluation(
    price_df_online,
    sentiment_series=sentiment_series,
    online_end_str=config.online_end,
    config=config,
    save_plots=True,
    plot_dir=result_picture_path(config),
    save_profile=True,
    profile_dir=result_profile_path(config),
    profile_name="ddpg_slm",
)

ddpg_slm_logs.head()


## 5. Output notes

- Figure files:
  - `addenda/result_picture/online_reward_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_wealth_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_daily_return_ddpg_slm_2026-01-01_2026-06-21.png`

- Profile file:
  - `addenda/result_profile_comparse/ddpg_slm_online_profile_2026-01-01_2026-06-21.csv`

- Compare it with the pure DDPG profile using `src/tool/compare_ddpg_profiles.py`.


## Workflow Notes - DDPG + SLM

- The SLM signal is a slow sentiment feature.

- It is not a tick-level trading signal.

- The DDPG model is trained without SLM first.

- During online evaluation, the environment reads:
  - the current trading date,
  - the daily sentiment value mapped from weekly news,
  - the same price-based technical state.

- The last observation slot stores the clipped sentiment score in `[-1, 1]`.

- This pipeline is experimental.

- The key question is simple:
  - Does a coarse sentiment feature improve the long-horizon behavior?
  - Or does it add noise to the investment policy?
